# Solar Power Generation – Prediction & Deviation Analysis
**Runs in Jupyter (Anaconda).** Put this notebook in the same folder as the `data/` folder
(containing the 4 Kaggle CSVs), or put the 4 CSVs next to the notebook. Run cells top to bottom.

Steps covered: 1 Problem → 2 Data → 3 Preprocessing → **4 EDA** → 5 Features → 6 Model selection → 7 Training → 8 Evaluation → **9 Hyperparameter tuning** → 10 Saving → **14 Testing**.
(Steps 11–13, 15–16 are the FastAPI app + website in the project folder.)

**Install once (Anaconda Prompt):** `pip install pandas scikit-learn matplotlib joblib`

## Step 1 – Problem definition
Predict plant-level AC power (sum over all inverters, every 15 min) from weather and time, then flag periods where actual output differs strongly from the prediction. Flags are **review points, not proof of equipment faults**.

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 4)

HERE = Path.cwd()
DATA = HERE / "data" if (HERE / "data").exists() else HERE
OUT = HERE / "outputs"; OUT.mkdir(exist_ok=True)
print("Reading CSVs from:", DATA)

## Step 2 – Data collection

In [ ]:
PLANTS = {
    "Plant 1": ("Plant_1_Generation_Data.csv", "Plant_1_Weather_Sensor_Data.csv", "%d-%m-%Y %H:%M"),
    "Plant 2": ("Plant_2_Generation_Data.csv", "Plant_2_Weather_Sensor_Data.csv", "%Y-%m-%d %H:%M:%S"),
}
raw = {}
for plant, (g, w, fmt) in PLANTS.items():
    gen, wx = pd.read_csv(DATA / g), pd.read_csv(DATA / w)
    raw[plant] = (gen, wx)
    print(f"{plant}: generation {gen.shape}, weather {wx.shape}")
raw["Plant 1"][0].head()

## Step 3 – Preprocessing
Parse dates (each plant uses a different format), drop bad rows, sum inverter power per timestamp, join weather on timestamp.

In [ ]:
chunks = []
for plant, (g, w, fmt) in PLANTS.items():
    gen, wx = raw[plant]
    gen = gen.copy(); wx = wx.copy()
    gen["timestamp"] = pd.to_datetime(gen["DATE_TIME"], format=fmt, errors="coerce")
    wx["timestamp"] = pd.to_datetime(wx["DATE_TIME"], errors="coerce")
    print(plant, "| bad dates:", gen.timestamp.isna().sum(), "| missing AC_POWER:", gen.AC_POWER.isna().sum(),
          "| duplicate weather stamps:", wx.timestamp.duplicated().sum())
    gen = gen.dropna(subset=["timestamp", "AC_POWER"]); wx = wx.dropna(subset=["timestamp"])
    power = gen.groupby("timestamp").agg(actual_ac_power=("AC_POWER", "sum"),
                                         zero_ac_inverters=("AC_POWER", lambda v: int((v == 0).sum()))).reset_index()
    wxm = wx.groupby("timestamp", as_index=False)[["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]].mean()
    m = power.merge(wxm, on="timestamp", how="inner", validate="one_to_one"); m["plant"] = plant
    chunks.append(m)
df = pd.concat(chunks, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
print("\nFinal table:", df.shape, "| range:", df.timestamp.min(), "->", df.timestamp.max())
df.describe().T[["count", "mean", "min", "max"]].round(2)

## Step 4 – Exploratory Data Analysis (EDA)

In [ ]:
# 4a. Data quality: how many 15-min intervals are missing per plant?
for p, g in df.groupby("plant"):
    full = pd.date_range(g.timestamp.min(), g.timestamp.max(), freq="15min")
    print(f"{p}: {len(g)} rows, {len(full) - len(g)} missing 15-min intervals ({100*(len(full)-len(g))/len(full):.1f}%)")
print("\nSummary by plant:"); display(df.groupby("plant")[["actual_ac_power", "IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE"]].describe().T.round(2))

In [ ]:
# 4b. Power over time
fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for a, (p, g) in zip(ax, df.groupby("plant")):
    a.plot(g.timestamp, g.actual_ac_power, lw=.7); a.set_title(f"{p} – total AC power"); a.set_ylabel("AC power")
plt.tight_layout(); plt.show()

In [ ]:
# 4c. Irradiation vs power (the main relationship) and average daily profile
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for p, g in df.groupby("plant"):
    ax[0].scatter(g.IRRADIATION, g.actual_ac_power, s=4, alpha=.4, label=p)
    prof = g.groupby(g.timestamp.dt.hour)[["actual_ac_power"]].mean(); ax[1].plot(prof.index, prof.actual_ac_power, label=p)
    ax[2].hist(g.actual_ac_power[g.IRRADIATION > .05], bins=40, alpha=.5, label=p)
ax[0].set(title="Irradiation vs AC power", xlabel="IRRADIATION", ylabel="AC power"); ax[1].set(title="Average power by hour", xlabel="hour")
ax[2].set(title="Daytime power distribution", xlabel="AC power"); [a.legend() for a in ax]; plt.tight_layout(); plt.show()

In [ ]:
# 4d. Correlations and zero-inverter behaviour
num = ["actual_ac_power", "IRRADIATION", "MODULE_TEMPERATURE", "AMBIENT_TEMPERATURE"]
corr = df[df.IRRADIATION > .05][num].corr()
plt.figure(figsize=(5, 4)); plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar()
plt.xticks(range(4), [c[:9] for c in num], rotation=45); plt.yticks(range(4), [c[:9] for c in num])
for i in range(4):
    for j in range(4): plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.title("Correlation (daytime only)"); plt.show()
day = df[df.IRRADIATION > .05]
print("Average inverters reading 0 power in DAYTIME:\n", day.groupby("plant").zero_ac_inverters.mean().round(2))

**EDA findings (from running this notebook on the provided data):**
- Irradiation is by far the strongest driver of AC power; module temperature is correlated mostly because it also rises with sunlight.
- Output is ~0 at night and peaks near midday – hour-of-day is an important feature.
- The data covers only about one month (15 May – 17 Jun 2020), so a *yearly* seasonal feature cannot be learned (day-of-year features were dropped).
- Plant 2 has noticeably more daytime scatter / zero-output inverters than Plant 1, so weather alone explains less of its output.

## Step 5 – Feature engineering
Cyclic hour-of-day (sin/cos), `temp_diff` = module − ambient temperature, plant one-hot. Removed: day-of-year features (no seasonal signal in one month).

In [ ]:
NUMERIC = ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION", "hour_sin", "hour_cos", "temp_diff"]
CATEGORICAL = ["plant"]; FEATURES = NUMERIC + CATEGORICAL; TARGET = "actual_ac_power"

def add_features(f):
    f = f.copy(); hour = f.timestamp.dt.hour + f.timestamp.dt.minute / 60
    f["hour_sin"] = np.sin(2 * np.pi * hour / 24); f["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    f["temp_diff"] = f.MODULE_TEMPERATURE - f.AMBIENT_TEMPERATURE
    return f
df = add_features(df).dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)

def make_pipeline(est, scale=False):
    prep = ColumnTransformer([("num", StandardScaler() if scale else "passthrough", NUMERIC),
                              ("plant", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return Pipeline([("features", prep), ("model", est)])

def metrics(y, p):
    return {"MAE": mean_absolute_error(y, p), "RMSE": mean_squared_error(y, p) ** .5, "R2": r2_score(y, p)}

# Chronological split 70 / 15 / 15 (never random – we must predict the *future*)
times = np.array(sorted(df.timestamp.unique())); a, b = int(len(times) * .70), int(len(times) * .85)
train = df[df.timestamp < times[a]]; val = df[(df.timestamp >= times[a]) & (df.timestamp < times[b])]; test = df[df.timestamp >= times[b]]
print("train", train.timestamp.min(), "→", train.timestamp.max(), len(train))
print("val  ", val.timestamp.min(), "→", val.timestamp.max(), len(val))
print("test ", test.timestamp.min(), "→", test.timestamp.max(), len(test))

## Step 6 – Model selection
Compare the models your project brief asks for (Linear Regression, SVR, Random Forest) plus Extra Trees and Gradient Boosting, all on the same validation period.

In [ ]:
std = lambda: TransformedTargetRegressor(SVR(C=10), transformer=StandardScaler())
candidates = {
    "Linear Regression": make_pipeline(LinearRegression(), scale=True),
    "SVR": make_pipeline(std(), scale=True),
    "Random Forest": make_pipeline(RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42)),
    "Extra Trees": make_pipeline(ExtraTreesRegressor(n_estimators=300, min_samples_leaf=4, max_features=.9, n_jobs=-1, random_state=42)),
    "Hist Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=250, learning_rate=.08, max_leaf_nodes=25, random_state=42)),
}
rows = {}
for name, m in candidates.items():                      # Step 7: training happens here
    m.fit(train[FEATURES], train[TARGET])
    rows[name] = metrics(val[TARGET], np.maximum(0, m.predict(val[FEATURES])))
pd.DataFrame(rows).T.sort_values("MAE").round(3)

## Step 9 – Hyperparameter tuning
`RandomizedSearchCV` with `TimeSeriesSplit` (each fold trains on the past and validates on the next block – no peeking at the future). Searched on train + validation only; the test period stays untouched.

In [ ]:
fit = pd.concat([train, val]).sort_values("timestamp").reset_index(drop=True)
param_dist = {"model__n_estimators": [200, 400, 600], "model__min_samples_leaf": [1, 2, 3, 4, 6, 8, 12],
              "model__max_features": [.5, .7, .9, 1.0], "model__max_depth": [None, 12, 18, 24]}
search = RandomizedSearchCV(make_pipeline(ExtraTreesRegressor(n_jobs=-1, random_state=42)), param_dist, n_iter=25,
                            cv=TimeSeriesSplit(n_splits=4), scoring="neg_mean_absolute_error", random_state=42, n_jobs=1, verbose=1)
search.fit(fit[FEATURES], fit[TARGET])
print("Best parameters:", search.best_params_)
print("Cross-validated MAE:", round(-search.best_score_, 1))

## Step 8 – Final evaluation on the unseen test period

In [ ]:
best = search.best_estimator_                       # already refit on train+validation
test = test.copy(); test["predicted"] = np.maximum(0, best.predict(test[FEATURES])); test["residual"] = test[TARGET] - test.predicted
day = test.IRRADIATION > .05
res = {"Both plants (all hours)": metrics(test[TARGET], test.predicted), "Both plants (daytime only)": metrics(test[TARGET][day], test.predicted[day])}
for p, g in test.groupby("plant"):
    res[f"{p} (all hours)"] = metrics(g[TARGET], g.predicted); gd = g[g.IRRADIATION > .05]; res[f"{p} (daytime only)"] = metrics(gd[TARGET], gd.predicted)
display(pd.DataFrame(res).T.round(3))
print("Note: 'all hours' includes easy night-time zeros, so daytime-only R² is the more honest number.")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for p, g in test.groupby("plant"):
    ax[0].scatter(g[TARGET], g.predicted, s=5, alpha=.5, label=p)
mx = test[TARGET].max(); ax[0].plot([0, mx], [0, mx], "k--"); ax[0].set(title="Actual vs predicted", xlabel="actual", ylabel="predicted"); ax[0].legend()
ax[1].hist(test.residual, bins=60); ax[1].set(title="Residual distribution")
g = test[test.plant == "Plant 2"]; ax[2].plot(g.timestamp, g[TARGET], label="actual", lw=.8); ax[2].plot(g.timestamp, g.predicted, label="predicted", lw=.8)
ax[2].set_title("Plant 2 – actual vs predicted"); ax[2].legend(); plt.tight_layout(); plt.show()

### Deviation detection (residual-based)
Threshold = 99th percentile of the *validation* absolute error per plant (using a model trained on train only, so validation errors are honest).

In [ ]:
val_model = make_pipeline(ExtraTreesRegressor(**{k.replace("model__", ""): v for k, v in search.best_params_.items()}, n_jobs=-1, random_state=42)).fit(train[FEATURES], train[TARGET])
vr = val.assign(pred=np.maximum(0, val_model.predict(val[FEATURES])))
thr = {p: float(np.quantile((g[TARGET] - g.pred).abs(), .99)) for p, g in vr.groupby("plant")}
test["threshold"] = test.plant.map(thr); test["flagged"] = test.residual.abs() > test.threshold
print("Thresholds:", {k: round(v, 1) for k, v in thr.items()}); print("Flagged test intervals:", test.groupby("plant").flagged.sum().to_dict())
display(test[test.flagged].sort_values("residual", key=abs, ascending=False)[["timestamp", "plant", TARGET, "predicted", "residual", "IRRADIATION", "zero_ac_inverters"]].head(10).round(2))
print("These are model discrepancies to review – NOT proof of equipment faults.")

## Step 10 – Save the model

In [ ]:
joblib.dump(best, OUT / "solar_model.joblib")
test[["timestamp", "plant", TARGET, "predicted", "residual", "threshold", "flagged"]].to_csv(OUT / "notebook_heldout_predictions.csv", index=False)
json.dump({"best_params": {k: (v if v is None or isinstance(v, (int, float)) else str(v)) for k, v in search.best_params_.items()},
           "test_metrics": res["Both plants (all hours)"]}, open(OUT / "notebook_metrics.json", "w"), indent=2, default=float)
print("Saved to", OUT)

## Step 14 – Testing
Automatic checks. If any line raises `AssertionError`, something is wrong.

In [ ]:
# Data tests
assert df[FEATURES + [TARGET]].isna().sum().sum() == 0, "missing values"
assert not df.duplicated(["plant", "timestamp"]).any(), "duplicate rows"
assert set(df.plant) == {"Plant 1", "Plant 2"}
# No leakage: splits are strictly chronological and the target is not a feature
assert train.timestamp.max() < val.timestamp.min() < test.timestamp.min()
assert TARGET not in FEATURES and "zero_ac_inverters" not in FEATURES
# Prediction tests
assert (test.predicted >= 0).all(), "negative power predicted"
assert test.predicted[test.IRRADIATION <= 0].max() < 50, "power predicted at night"
assert res["Both plants (all hours)"]["R2"] > .93 and res["Both plants (all hours)"]["MAE"] < 800, "accuracy too low"
# Saved model reproduces the same predictions after reloading
reloaded = joblib.load(OUT / "solar_model.joblib")
assert np.allclose(np.maximum(0, reloaded.predict(test[FEATURES])), test.predicted), "reloaded model differs"
# Behaviour tests with hand-made inputs
def predict_one(plant, ts, amb, mod, irr):
    row = add_features(pd.DataFrame([{"timestamp": pd.Timestamp(ts), "plant": plant, "AMBIENT_TEMPERATURE": amb, "MODULE_TEMPERATURE": mod, "IRRADIATION": irr}]))
    return float(max(0, reloaded.predict(row[FEATURES])[0]))
noon, dark = predict_one("Plant 1", "2020-06-10 12:00", 32, 50, .8), predict_one("Plant 1", "2020-06-10 02:00", 24, 23, 0)
assert noon > 5000 and dark < 50 and noon > predict_one("Plant 1", "2020-06-10 12:00", 30, 40, .3), "physics sanity failed"
print("✅ All tests passed  |  noon estimate:", round(noon), " night estimate:", round(dark))

## Next: Steps 11–16
The FastAPI backend (`api.py`), website (`static/index.html`), API tests (`tests/test_model.py`) and the run instructions are in the project folder. Run `python -m pytest -v` there to test the API as well.